In [51]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns;
import pandas as pd

In [52]:
mel_data = (
    pd.read_excel('Melbourne01.xlsx')
    .rename(columns=lambda col: col.strip())
    .dropna()
)
mel_data.head()

,Year,Month,Day,Hour,Min,Air Temp (degrees C),Apparent Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),Wind Gust (km/h),MSLP (hPa),Rainfall since 9 am (mm),gamma,Calculated Dew Pt Temp (degrees C),E (hPa),Calculated Apparent Temp (degrees C)
0,2011,1,1.0,0.0,4.0,24.8,0.0,14.0,51.0,SE,11,13.0,1007.4,0,0.969609,14.1,15.916676,23.9
1,2011,1,1.0,0.0,14.0,24.8,0.0,13.3,48.0,SE,11,11.0,1007.5,0,0.908985,13.2,14.980401,23.6
2,2011,1,1.0,0.0,24.0,24.9,0.0,13.3,48.0,SE,11,13.0,1007.5,0,0.915025,13.2,15.069879,23.7
3,2011,1,1.0,0.0,34.0,24.7,0.0,13.4,49.0,SE,11,11.0,1007.4,0,0.923560,13.4,15.201624,23.6
4,2011,1,1.0,0.0,44.0,24.1,0.0,13.3,51.0,ESE,9,9.0,1007.3,0,0.927209,13.4,15.264860,23.4


In [53]:
numeric_cols = mel_data.columns.drop("Wind Direction")

mel_data[numeric_cols] = mel_data[numeric_cols].apply(
    pd.to_numeric, errors='coerce'
)

mel_data = mel_data.dropna(subset=numeric_cols)

mel_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 504500 entries, 0 to 505354
Data columns (total 18 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Year                                  504500 non-null  int64  
 1   Month                                 504500 non-null  int64  
 2   Day                                   504500 non-null  float64
 3   Hour                                  504500 non-null  float64
 4   Min                                   504500 non-null  float64
 5   Air Temp (degrees C)                  504500 non-null  float64
 6   Apparent Temp (degrees C)             504500 non-null  float64
 7   Dew Pt Temp (degrees C)               504500 non-null  float64
 8   Humidity (%)                          504500 non-null  float64
 9   Wind Direction                        504500 non-null  object 
 10  Wind Speed (km/h)                     504500 non-null  int64  
 11  Wind 

In [77]:
mel_data.rename(columns={'Min': 'Minute'}, inplace=True)

mel_data['Datetime'] = pd.to_datetime(
    mel_data[['Year','Month','Day','Hour','Minute']]
)
mel_data.sort_values('Datetime', inplace=True)
mel_data.Datetime.head()

0   2011-01-01 00:04:00
1   2011-01-01 00:14:00
2   2011-01-01 00:24:00
3   2011-01-01 00:34:00
4   2011-01-01 00:44:00
Name: Datetime, dtype: datetime64[ns]

In [78]:
temp_cols = ['Air Temp (degrees C)', 'Apparent Temp (degrees C)', 'Calculated Apparent Temp (degrees C)']
for col in temp_cols:
    mel_data = mel_data[(mel_data[col] >= -20) & (mel_data[col] <= 55)]

dew_cols = ['Dew Pt Temp (degrees C)', 'Calculated Dew Pt Temp (degrees C)']
for col in dew_cols:
    mel_data = mel_data[mel_data[col] >= -20]
    mel_data = mel_data[mel_data[col] <= mel_data['Air Temp (degrees C)']]

mel_data = mel_data[(mel_data['Humidity (%)'] >= 0) & (mel_data['Humidity (%)'] <= 100)]

mel_data = mel_data[mel_data['Wind Speed (km/h)'] >= 0]
mel_data = mel_data[(mel_data['Wind Gust  (km/h)'] >= 0) & (mel_data['Wind Gust  (km/h)'] <= 120)]

mel_data = mel_data[(mel_data['MSLP (hPa)'] >= 950) & (mel_data['MSLP (hPa)'] <= 1050)]

mel_data = mel_data[(mel_data['E (hPa)'] >= 5) & (mel_data['E (hPa)'] <= 30)]

mel_data = mel_data[(mel_data['gamma'] >= 0.5) & (mel_data['gamma'] <= 2.5)]

mel_data = mel_data[mel_data['Rainfall since 9 am (mm)'] >= 0]

In [83]:
mel_data.resample('D', on='Datetime').agg({
    'Air Temp (degrees C)': 'mean',
    'Humidity (%)': 'mean',
    'MSLP (hPa)': 'mean',
    'Wind Speed (km/h)': 'mean',
    'Wind Gust  (km/h)': 'max',
    'E (hPa)': 'mean',
    'Dew Pt Temp (degrees C)': 'mean',
    'Apparent Temp (degrees C)': 'mean',
    'Rainfall since 9 am (mm)': 'sum'
})

,Air Temp (degrees C),Humidity (%),MSLP (hPa),Wind Speed (km/h),Wind Gust (km/h),E (hPa),Dew Pt Temp (degrees C),Apparent Temp (degrees C),Rainfall since 9 am (mm)
Datetime,,,,,,,,,
2011-01-01,19.651938,61.635659,1011.784496,14.325581,43.0,13.980619,12.004651,0.000000,0.0
2011-01-02,18.560563,53.718310,1015.983099,22.704225,39.0,11.300188,8.839437,0.000000,0.0
2011-01-03,17.300000,51.000000,1016.400000,26.000000,35.0,10.048411,7.100000,0.000000,0.0
2011-01-04,17.183478,59.608696,1012.045217,15.782609,37.0,11.574564,9.133043,0.000000,0.0
2011-01-05,18.110853,66.565891,1010.086047,19.837209,37.0,13.699222,11.719380,0.000000,0.0
...,...,...,...,...,...,...,...,...,...
2022-03-04,24.486400,66.512000,1009.955200,15.688000,46.0,19.023386,16.672800,25.424800,20.6
2022-03-05,19.320301,89.759398,1007.534586,19.406015,43.0,20.054762,17.533835,20.012030,1349.4
2022-03-06,18.484615,79.061538,1015.901538,27.607692,57.0,16.734805,14.642308,17.083077,188.6


In [84]:
mel_data.resample('M', on='Datetime').agg({
    'Air Temp (degrees C)': 'mean',
    'Humidity (%)': 'mean',
    'MSLP (hPa)': 'mean',
    'Wind Speed (km/h)': 'mean',
    'Wind Gust  (km/h)': 'max',
    'E (hPa)': 'mean',
    'Dew Pt Temp (degrees C)': 'mean',
    'Apparent Temp (degrees C)': 'mean',
    'Rainfall since 9 am (mm)': 'sum'
})

/var/folders/q0/1kjs8s8d6fn65050zv_j1x7c0000gn/T/ipykernel_73783/1238222465.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  mel_data.resample('M', on='Datetime').agg({


,Air Temp (degrees C),Humidity (%),MSLP (hPa),Wind Speed (km/h),Wind Gust (km/h),E (hPa),Dew Pt Temp (degrees C),Apparent Temp (degrees C),Rainfall since 9 am (mm)
Datetime,,,,,,,,,
2011-01-31,21.447324,64.843805,1011.050553,16.987493,63.0,16.389007,14.041187,0.000000,4297.4
2011-02-28,20.933896,67.639032,1014.094497,15.574101,65.0,16.571908,14.316618,0.000000,1954.2
2011-03-31,19.213039,65.967888,1015.366591,17.373986,72.0,14.488432,12.261953,0.000000,2066.0
2011-04-30,16.358959,68.659567,1022.634990,16.574375,56.0,12.379933,10.124172,0.000000,3866.8
2011-05-31,13.321645,77.299229,1018.192596,14.741902,67.0,11.745212,9.316812,0.000000,2963.2
...,...,...,...,...,...,...,...,...,...
2021-11-30,16.789463,71.755181,1014.791113,18.714085,82.0,13.284986,10.980295,15.214366,5652.2
2021-12-31,18.740937,63.429983,1014.454433,19.250423,80.0,13.216647,10.999520,17.042377,1994.2
2022-01-31,22.901135,64.495558,1012.425642,17.572310,65.0,17.445545,15.166214,22.946496,6212.4
